# Climatology Virtual Dataset with Kerchunk

This notebook demonstrates how to create a virtual climatology dataset that:
- Uses the full date range from a daily dataset
- Maps each date to its corresponding day-of-year climatological average
- Uses kerchunk references to avoid data duplication

In [1]:
import numpy as np
import xarray as xr
import dask.array as da
from virtualizarr import open_virtual_dataset
import pandas as pd
import json
import fsspec
from virtualizarr.parsers import ZarrParser
from obstore.store import LocalStore
from virtualizarr.registry import ObjectStoreRegistry
import nest_asyncio

nest_asyncio.apply()

## 1. Create Example Dataset

Create a multi-year daily dataset (2000-2003) with random temperature data.

In [2]:
# Create example data with Dask arrays
time = pd.date_range(start="2000-01-01", end="2003-12-31", freq="D")
x = np.arange(5)  # 5 x points
y = np.arange(3)  # 3 y points

# Create dask array with time chunked to 1
data = da.random.random((len(time), len(x), len(y)))

# Create xarray Dataset with dask arrays
ds = xr.Dataset(
    {
        "temperature": (["time", "x", "y"], data),
    },
    coords={
        "time": time,
        "x": x,
        "y": y,
    },
)

print(ds)

<xarray.Dataset> Size: 187kB
Dimensions:      (time: 1461, x: 5, y: 3)
Coordinates:
  * time         (time) datetime64[us] 12kB 2000-01-01 2000-01-02 ... 2003-12-31
  * x            (x) int64 40B 0 1 2 3 4
  * y            (y) int64 24B 0 1 2
Data variables:
    temperature  (time, x, y) float64 175kB dask.array<chunksize=(1461, 5, 3), meta=np.ndarray>


## 2. Save Datasets to Zarr

Save the daily dataset and create a day-of-year climatology (averaged by day of year).

In [3]:
output_path = "example_data.zarr"

In [4]:
# Define chunking: time=1, x=50, y=30 (full x and y, but time chunked by 1)
ds.to_zarr(output_path, mode="w", encoding={"temperature": {"chunks": (1, 50, 30)}})

print(f"Dataset saved to {output_path}")
print("Chunking: time=1, x=50, y=30")

/Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Dataset saved to example_data.zarr
Chunking: time=1, x=50, y=30


## 3. Create Day-of-Year Climatology

Group by day of year and calculate mean across all years.

In [5]:
ds_averaged = ds.groupby("time.dayofyear").mean("time")
ds_averaged

<xarray.Dataset> Size: 47kB
Dimensions:      (dayofyear: 366, x: 5, y: 3)
Coordinates:
  * dayofyear    (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
  * x            (x) int64 40B 0 1 2 3 4
  * y            (y) int64 24B 0 1 2
Data variables:
    temperature  (dayofyear, x, y) float64 44kB dask.array<chunksize=(1, 5, 3), meta=np.ndarray>

In [6]:
ds_averaged.to_zarr(
    "example_data_averaged.zarr",
    mode="w",
    encoding={"temperature": {"chunks": (1, 50, 30)}},
)

/Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


## 4. Create Virtual Datasets with VirtualiZarr

Load both datasets as virtual datasets and extract their kerchunk references.

In [7]:
zarr_store = "/Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/src/" + str(
    output_path
)
store = LocalStore(prefix=zarr_store)
registry = ObjectStoreRegistry({f"file://{zarr_store}": store})
parser = ZarrParser()
vds = open_virtual_dataset(url=zarr_store, registry=registry, parser=parser)
refs = vds.vz.to_kerchunk()

## 5. Build Climatology References

Load the averaged dataset references and create a new refs dictionary that maps each date in the full time series to its corresponding day-of-year climatology.

In [8]:
# Load the averaged dataset to get its references
zarr_store_avg = (
    "/Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/src/example_data_averaged.zarr"
)
store_avg = LocalStore(prefix=zarr_store_avg)
registry_avg = ObjectStoreRegistry({f"file://{zarr_store_avg}": store_avg})
vds_avg = open_virtual_dataset(url=zarr_store_avg, registry=registry_avg, parser=parser)
refs_avg = vds_avg.vz.to_kerchunk()

print("Averaged dataset refs loaded")
print(
    f"Number of dayofyear chunks: {len([k for k in refs_avg['refs'].keys() if k.startswith('temperature/') and '.0.0' in k])}"
)

Averaged dataset refs loaded
Number of dayofyear chunks: 366


In [9]:
# Create new refs dict based on ds time dimension but referencing ds_averaged chunks
new_refs = {"version": 1, "refs": {}}

# Copy basic metadata
new_refs["refs"][".zgroup"] = refs["refs"][".zgroup"]
new_refs["refs"][".zattrs"] = refs["refs"][".zattrs"]

# Copy x and y (spatial dimensions remain the same)
for key in refs["refs"].keys():
    if key.startswith("x/") or key.startswith("y/"):
        new_refs["refs"][key] = refs["refs"][key]

# Get the original time coordinate from ds
time_array_metadata = json.loads(refs["refs"]["time/.zarray"])
n_times = time_array_metadata["shape"][0]  # Total number of time steps in ds

# Get day of year for each time step
time_coords = ds.time.values
dayofyear = pd.DatetimeIndex(time_coords).dayofyear

print(f"Total time steps in ds: {n_times}")
print(f"Day of year range: {dayofyear.min()} to {dayofyear.max()}")
print(f"Number of unique day of years: {len(np.unique(dayofyear))}")

Total time steps in ds: 1461
Day of year range: 1 to 366
Number of unique day of years: 366


In [10]:
# Keep the original time coordinate from ds (full date range)
new_refs["refs"]["time/0"] = refs["refs"]["time/0"]
new_refs["refs"]["time/.zarray"] = refs["refs"]["time/.zarray"]
new_refs["refs"]["time/.zattrs"] = refs["refs"]["time/.zattrs"]

# Map temperature chunks: each time step references the corresponding dayofyear chunk
for time_idx in range(n_times):
    # Get the day of year for this time step (1-indexed in pandas, but 0-indexed in zarr)
    doy = dayofyear[time_idx]

    # dayofyear is 1-indexed (1-366), but zarr chunks are 0-indexed
    # So doy 1 maps to chunk 0, doy 2 maps to chunk 1, etc.
    doy_chunk_idx = doy - 1

    # Reference the corresponding dayofyear chunk from ds_averaged
    new_key = f"temperature/{time_idx}.0.0"
    avg_key = f"temperature/{doy_chunk_idx}.0.0"

    if avg_key in refs_avg["refs"]:
        new_refs["refs"][new_key] = refs_avg["refs"][avg_key]
    else:
        print(f"Warning: {avg_key} not found in averaged refs")

# Update temperature metadata with new time dimension size
temp_array_metadata = json.loads(refs_avg["refs"]["temperature/.zarray"])
temp_array_metadata["shape"] = [
    n_times,
    temp_array_metadata["shape"][1],
    temp_array_metadata["shape"][2],
]

new_refs["refs"]["temperature/.zarray"] = json.dumps(temp_array_metadata)

# Update temperature attributes to use 'time' dimension instead of 'dayofyear'
temp_attrs = json.loads(refs_avg["refs"]["temperature/.zattrs"])
temp_attrs["_ARRAY_DIMENSIONS"] = ["time", "x", "y"]  # Rename dayofyear to time
new_refs["refs"]["temperature/.zattrs"] = json.dumps(temp_attrs)

print("\nNew refs created:")
print(f"  Time dimension: {n_times} steps (full date range)")
print(f"  Temperature shape: {temp_array_metadata['shape']}")
print("  Temperature chunks mapped from ds_averaged based on day of year")


New refs created:
  Time dimension: 1461 steps (full date range)
  Temperature shape: [1461, 5, 3]
  Temperature chunks mapped from ds_averaged based on day of year


In [11]:
# Write the new refs to JSON
refs_json_path = (
    "/Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/src/climatology_refs.json"
)

with open(refs_json_path, "w") as f:
    json.dump(new_refs, f, indent=2)

print(f"Climatology refs written to: {refs_json_path}")

Climatology refs written to: /Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/src/climatology_refs.json


In [12]:
# Open the climatology dataset with xarray

fs = fsspec.filesystem("reference", fo=refs_json_path)
mapper = fs.get_mapper("")

ds_climatology = xr.open_dataset(
    mapper, engine="zarr", backend_kwargs={"consolidated": False}
)
ds_climatology

<xarray.Dataset> Size: 187kB
Dimensions:      (time: 1461, x: 5, y: 3)
Coordinates:
  * time         (time) datetime64[ns] 12kB 2000-01-01 2000-01-02 ... 2003-12-31
  * x            (x) int64 40B 0 1 2 3 4
  * y            (y) int64 24B 0 1 2
Data variables:
    temperature  (time, x, y) float64 175kB ...

In [13]:
# Check the actual structure
print("ds_climatology dimensions:", ds_climatology.dims)
print("ds_climatology coords:", list(ds_climatology.coords.keys()))
print("\nds_averaged dimensions:", ds_averaged.dims)
print("ds_averaged coords:", list(ds_averaged.coords.keys()))

ds_climatology dimensions: FrozenMappingWarningOnValuesAccess({'time': 1461, 'x': 5, 'y': 3})
ds_climatology coords: ['x', 'y', 'time']

ds_averaged dimensions: FrozenMappingWarningOnValuesAccess({'dayofyear': 366, 'x': 5, 'y': 3})
ds_averaged coords: ['x', 'y', 'dayofyear']


## 6. Save and Load Climatology Dataset

Write the climatology references to JSON and open as an xarray dataset using fsspec.

In [14]:
# Verify: Check that same day of year across different years has the same values
jan_1_2000 = ds_climatology.temperature.sel(time="2000-01-01").values
jan_1_2001 = ds_climatology.temperature.sel(time="2001-01-01").values
jan_1_2002 = ds_climatology.temperature.sel(time="2002-01-01").values

print("Verification: Same day of year should have same values across years")
print(f"Jan 1, 2000 equals Jan 1, 2001: {np.allclose(jan_1_2000, jan_1_2001)}")
print(f"Jan 1, 2000 equals Jan 1, 2002: {np.allclose(jan_1_2000, jan_1_2002)}")

# Check different day
feb_29_2000 = ds_climatology.temperature.sel(time="2000-02-29").values  # leap year
feb_29_2004 = (
    ds_climatology.temperature.sel(time="2004-02-29")
    if "2004-02-29" in ds.time.dt.strftime("%Y-%m-%d").values
    else None
)
if feb_29_2004 is not None:
    print("Feb 29, 2000 would equal Feb 29, 2004: both are day 60")

print(
    "\n✓ Climatology mapping confirmed: Each date references its corresponding day-of-year from ds_averaged!"
)

Verification: Same day of year should have same values across years
Jan 1, 2000 equals Jan 1, 2001: True
Jan 1, 2000 equals Jan 1, 2002: True

✓ Climatology mapping confirmed: Each date references its corresponding day-of-year from ds_averaged!
